In [43]:
import copy
from tqdm import tqdm
from torch.autograd import Variable
import argparse
import time
import os
from src.language_models.dictionary_corpus import Dictionary, tokenize
import pandas
import torch
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict



In [15]:
def feed_input(model, hidden, w):
    inp = torch.autograd.Variable(torch.LongTensor([[vocab.word2idx[w]]]))
    
    out, hidden = model(inp, hidden)
    return out, hidden


def feed_sentence(model, h, sentence):
    outs = []
    for w in sentence:
        out, h = feed_input(model, h, w)
        outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))
    return outs, h

In [3]:
vocab = Dictionary("/scratch2/mrenaudin/colorlessgreenRNNs/english_data")

In [ ]:
input = "/scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp"


In [5]:
sentences = [
    l.rstrip("\n").split(" ") for l in open("/scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.text", encoding="utf-8")
]

In [7]:
sentences[0]

['The', 'athlete', 'behind', 'the', 'bike', 'observes']

In [8]:
gold = pandas.read_csv(
    "/scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.gold",
    sep="\t",
    header=None,
    names=["verb_pos", "correct", "wrong", "nattr"],
)

In [9]:
gold

,verb_pos,correct,wrong,nattr
0,5,observes,observe,-999
1,5,discourages,discourage,-999
2,5,encourages,encourage,-999
3,5,avoids,avoid,-999
4,5,confuses,confuse,-999
...,...,...,...,...
3995,5,encourage,encourages,-999
3996,5,remember,remembers,-999
3997,5,stimulate,stimulates,-999
3998,5,inspire,inspires,-999


In [10]:
different_values = gold[gold['verb_pos'] != 5]

# Display unique values different from 5
print(different_values['verb_pos'].unique())

[]


In [11]:
from src.language_models.model import RNNModel as lstm

print("\nmodel: " + "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt" + "\n")
model = lstm("LSTM", 50001, 200, 650, 2, 0.2, False)
with open("/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt", "rb") as f:
    print("Loading the model")
    state_dict = torch.load(f, map_location="cpu")
    model.load_state_dict(state_dict)



model: /scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt

Loading the model


## Exp 1 :  compare matrices of probabilities before choosing which proba to compare

In [216]:
model_orig_state = copy.deepcopy(model.state_dict())

log_p_targets_correct = np.zeros((len(sentences), 1))
log_p_targets_wrong = np.zeros((len(sentences), 1))


model.load_state_dict(model_orig_state)
model.eval()

RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 200)
  (rnn): LSTM(200, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)

In [217]:
init_sentence = " ".join(
        [
            "In service , the aircraft was operated by a crew of five and could accommodate either 30 paratroopers , 32 <unk> and 28 sitting casualties , or 50 fully equipped troops . <eos>",
            'He even speculated that technical classes might some day be held " for the better training of workmen in their several crafts and industries . <eos>',
            "After the War of the Holy League in 1537 against the Ottoman Empire , a truce between Venice and the Ottomans was created in 1539 . <eos>",
            'Moore says : " Tony and I had a good <unk> and off-screen relationship , we are two very different people , but we did share a sense of humour " . <eos>',
            "<unk> is also the basis for online games sold through licensed lotteries . <eos>",
        ]
)

In [218]:
model.eval()
hidden = model.init_hidden(1)
init_out, init_h = feed_sentence(model, hidden, init_sentence.split(" "))


/tmp/ipykernel_1357910/2337286375.py:12: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))


In [247]:

log_p_targets_correct = np.zeros((len(sentences), 1))
log_p_targets_wrong = np.zeros((len(sentences), 1))

In [265]:
for i, s in enumerate(tqdm(sentences)):
    out = None
    hidden = init_h  # model.init_hidden(1)
   
    for k, w in enumerate(s):
        inp = Variable(torch.LongTensor([[vocab.word2idx[w]]]))
        out2, hidden = model(inp, hidden)
        out2 = torch.nn.functional.log_softmax(out2[0]).unsqueeze(0)
        if k == gold.loc[i, "verb_pos"] - 1:
            print(gold.loc[i, "verb_pos"] - 1)
            assert s[k + 1] == gold.loc[i, "correct"].lower()
            print(vocab.word2idx[gold.loc[i, "correct"]])
            # Store surprisal of target word
            log_p_targets_correct[i] = out2[
                0, 0, vocab.word2idx[gold.loc[i, "correct"]]
            ].data.item()
            print(log_p_targets_correct[i])
            log_p_targets_wrong[i] = out2[
                0, 0, vocab.word2idx[gold.loc[i, "wrong"]]
            ].data.item()
    break 

  0%|          | 0/4000 [00:00<?, ?it/s]/tmp/ipykernel_1357910/663550449.py:8: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  out2 = torch.nn.functional.log_softmax(out2[0]).unsqueeze(0)
  0%|          | 0/4000 [00:00<?, ?it/s]

4
8739
[-11.67110062]


In [220]:
out2.shape #proba over the vocabulary for the last word

torch.Size([1, 1, 50001])

## Now with the dataloader

In [221]:
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"  # current directory
dictionary = Dictionary(data_path)
checkpoint_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt"  # Replace with your checkpoint path
nounpp = '//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt'

In [222]:
class NounPPDataset(Dataset):
    def __init__(self, nounpp_file, dictionary):
        self.sentences = []
        self.conditions = []
        self.correct = []
        self.wrong = []
        self.encoded_sentences=[]
        self.encoded_correct = []
        self.encoded_wrong = []
        self.dictionary = dictionary

        with open(nounpp_file, "r") as f:
            for line in f:
                line = line.split()
                sentence = line[1:7]
                condition = " ".join(line[7:9])
                wrong = line[9]
                correct = line[6]
                encoded_sentence = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence]
                encoded_correct = self.dictionary.word2idx.get(correct, self.dictionary.word2idx.get("<unk>"))
                encoded_wrong = self.dictionary.word2idx.get(wrong, self.dictionary.word2idx.get("<unk>"))
                
                self.sentences.append(sentence)
                self.conditions.append(condition)
                self.correct.append(correct)
                self.wrong.append(wrong)
                self.encoded_sentences.append(encoded_sentence)
                self.encoded_correct.append(encoded_correct)
                self.encoded_wrong.append(encoded_wrong)

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return {
            "sentence": self.sentences[idx],
            "encoded_sentence": torch.tensor(self.encoded_sentences[idx], dtype=torch.long),
            "correct": self.correct[idx],
            "encoded_correct":torch.tensor(self.encoded_correct[idx], dtype = torch.long),
            "wrong":self.wrong[idx],
            "encoded_wrong":torch.tensor(self.encoded_wrong[idx], dtype = torch.long),
            "condition": self.conditions[idx],
        }

In [223]:
test_dataset = NounPPDataset(nounpp, dictionary)

In [224]:
def collate_fn(batch):
    """Custom collate function to properly handle sentences as lists of strings."""
    sentences = [item['sentence'] for item in batch]  # Keep lists of words as they are
    encoded_sentences = torch.stack([item['encoded_sentence'] for item in batch])  # Stack tensors
    encoded_correct = torch.stack([item['encoded_correct'] for item in batch])
    encoded_wrong = torch.stack([item['encoded_wrong'] for item in batch])
    correct = [item['correct'] for item in batch]
    wrong = [item['wrong'] for item in batch]
    conditions = [item['condition'] for item in batch]
    
    return {
        "sentence": sentences,  
        "encoded_sentence": encoded_sentences,
        "correct": correct,
        "encoded_correct": encoded_correct,
        "wrong": wrong,
        "encoded_wrong": encoded_wrong,
        "condition": conditions
    }
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fn)

In [225]:
model.eval()
hidden = model.init_hidden(1) 
init_out1, init_h1 = feed_sentence(model, hidden, init_sentence.split(" "))



/tmp/ipykernel_1357910/2337286375.py:12: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  outs.append(torch.nn.functional.log_softmax(out[0]).unsqueeze(0))


In [267]:
condition_accuracies = defaultdict(int)
condition_counts = defaultdict(int)
correct_pred = 0
sentence_details = []

s = torch.nn.LogSoftmax(dim=-1)
model.eval()
#Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in test_dataloader:
        out=None
        written = batch['sentence']
        sentence = batch['encoded_sentence']
        correct = batch['encoded_correct']
        wrong = batch['encoded_wrong']
        condition = batch['condition']
        batch_size = sentence.size(0)
        hidden = (init_h[0].expand(-1, batch_size, -1).contiguous(), 
          init_h[1].expand(-1, batch_size, -1).contiguous()) 
      
        for w in range(sentence.shape[1]-1):#update hidden state word by word
            # for w in range(len(sen)):
                
            word = torch.autograd.Variable(sentence[:,w].unsqueeze(0))
            out1, hidden = model(word, hidden)
        log_probs= torch.nn.functional.log_softmax(out1, dim=-1)
        print(correct[0])
        correct_log_probs = log_probs[0, torch.arange(batch_size), correct]  # Shape: [512]
        wrong_log_probs = log_probs[0, torch.arange(batch_size), wrong]
            # out1=out1[:,0,:].unsqueeze(0) 
            # out1= torch.nn.functional.log_softmax(out1[0]).unsqueeze(0)

        break

tensor(8739)


In [272]:
correct_log_probs[0]

tensor(-11.6711)

In [269]:
out2.shape

torch.Size([1, 1, 50001])

In [270]:
out2[0,0,8739]

tensor(-14.2948, grad_fn=<SelectBackward0>)